# E006 — Assemble & Validate V2 Submission

Run this **after** the modeling/search notebooks you intend to promote.

## Kaggle input checklist
- **Required:** repository source and preferably E004 `learned_model.json`
- **Recommended:** E007 `meta_artifact.json` (built from E005 policy-zoo results)
- **Fallback:** E005 `cem_best.json` if no meta artifact exists
- **Accelerator:** None / CPU
- **Internet:** Off
- **Output:** `submission_v2.tar.gz`

Active market probes are never enabled merely because threshold research exists. They require a separate paired promotion decision.


In [ ]:
from pathlib import Path
import sys,json,shutil,tarfile,py_compile
roots=[Path('/kaggle/input'),Path('/kaggle/working'),Path.cwd().parent,Path.cwd()]
repo_file=next((p for r in roots for p in r.rglob('submission/base_controller.py')),None)
if repo_file is None: raise FileNotFoundError('Attach the kaggriculture repository')
ROOT=repo_file.parents[1];sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'src'))
WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts';WORK.mkdir(parents=True,exist_ok=True)
SUB=ROOT/'submission';print('ROOT',ROOT,'WORK',WORK)
def find_one(name):
    hits=[p for r in roots for p in r.rglob(name)]
    return hits[0] if hits else None


In [ ]:
learned=find_one('learned_model.json')
# Prefer an external E004 artifact over the checked-in empty scaffold.
external=[p for r in roots[:2] for p in r.rglob('learned_model.json') if 'submission' not in p.parts]
if external: learned=external[0]
model=json.loads(learned.read_text()) if learned else {'version':2}
model['version']=max(2,int(model.get('version',0)))
meta_path=find_one('meta_artifact.json')
if meta_path:
    model['meta']=json.loads(meta_path.read_text());print('embedded meta',meta_path)
elif find_one('cem_best.json'):
    best=json.loads(find_one('cem_best.json').read_text())['best'];n=len((model.get('archetype') or {}).get('centroids',[]));model['policy_by_archetype']={str(i):best for i in range(n)}
    print('using legacy archetype-wide CEM fallback')
# Probe research stays disabled unless a future explicitly promoted artifact says otherwise.
model.setdefault('probe',{'enabled':False})
(SUB/'learned_model.json').write_text(json.dumps(model,indent=2,sort_keys=True))
files=['main.py','predictive_agent.py','parametric_agent.py','base_controller.py','runtime_model.py','meta_runtime.py']
for f in files: py_compile.compile(str(SUB/f),doraise=True)
print('model bytes',(SUB/'learned_model.json').stat().st_size)


In [ ]:
from submission.predictive_agent import PredictiveMind
from submission.base_controller import HarvestMind
from src.kagv2.simulator import Game
for seed in range(3):
    cash=Game(seed=seed).run([PredictiveMind().act,HarvestMind().act]);print('smoke',seed,cash)


In [ ]:
OUT=WORK/'submission_v2.tar.gz';files=['main.py','predictive_agent.py','parametric_agent.py','base_controller.py','runtime_model.py','meta_runtime.py','learned_model.json']
with tarfile.open(OUT,'w:gz') as t:
    for f in files:t.add(SUB/f,arcname=f)
with tarfile.open(OUT) as t:names=t.getnames()
assert 'main.py' in names and all('/' not in n for n in names)
print('READY',OUT,'bytes',OUT.stat().st_size,'files',names)


## Before spending a ladder submission
1. Run both-seat held-out tournaments against V1 and the policy zoo.
2. Require robust/worst-archetype metrics, not only mean score.
3. If `kaggle_environments` is available, run official-engine parity.
4. Profile the agent comfortably below the 1-second action limit.
5. Submit one promoted candidate, then mine its hosted episodes before iterating.
